## Setup

In [1]:
import os
import sys
import time
import logging
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, str(Path("../").resolve()))

from src.part1.bash_tool import run_bash
from src.part1.classifier import classify_query
from src.part1.retriever import retrieve_context
from src.part1.pipeline import build_pipeline, answer_question

load_dotenv(dotenv_path="../.env")

logging.basicConfig(
    format="[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

/Users/qing0304/Desktop/GU/DSAN6725/spring-2026-a03-Cynthia2734/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Verify API key is set
if os.getenv("GROQ_API_KEY"):
    logger.info("GROQ_API_KEY is set")
elif os.getenv("OPENAI_API_KEY"):
    logger.info("OPENAI_API_KEY is set")
elif os.getenv("ANTHROPIC_API_KEY"):
    logger.info("ANTHROPIC_API_KEY is set")
else:
    logger.warning("No API key found! Please set GROQ_API_KEY in your .env file")

# Check for Cohere API key (needed for re-ranking)
if os.getenv("COHERE_API_KEY"):
    logger.info("COHERE_API_KEY is set (for re-ranking)")
else:
    logger.warning("COHERE_API_KEY not set - re-ranking section will not work")

[2026-03-04 23:42:56,331] p37301 {21338686.py:3} INFO - GROQ_API_KEY is set
[2026-03-04 23:42:56,331] p37301 {21338686.py:13} INFO - COHERE_API_KEY is set (for re-ranking)


## Configuration

In [3]:
CODEBASE_PATH = "../mcp-gateway-registry"
MODEL_ID = "groq/llama-3.1-8b-instant"

# Build pipeline
pipeline = build_pipeline(model_id=MODEL_ID, codebase_path=CODEBASE_PATH)

test_questions = [
    "What Python dependencies does this project use?",
    "What is the main entry point file for the registry service?",
    "What programming languages and file types are used in this repository?",
    "How does the authentication flow work, from token validation to user authorization?",
    "What are all the API endpoints available in the registry service and what scopes do they require?",
    "How would you add support for a new OAuth provider (e.g., Okta) to the authentication system?",
]

/Users/qing0304/Desktop/GU/DSAN6725/spring-2026-a03-Cynthia2734/src/part1/pipeline.py:53: LangChainDeprecationWarning: The class `ChatLiteLLM` was deprecated in LangChain 0.3.24 and will be removed in 1.0. An updated version of the class exists in the `langchain-litellm package and should be used instead. To use it run `pip install -U `langchain-litellm` and import as `from `langchain_litellm import ChatLiteLLM``.
  self.llm = ChatLiteLLM(model=model_id, temperature=temperature)
[2026-03-04 23:42:57,278] p37301 {pipeline.py:54} INFO - Part1Pipeline initialised (model=groq/llama-3.1-8b-instant)


## Verify Bash Tool

In [4]:
output = run_bash("pwd", cwd=CODEBASE_PATH)
logger.info(output)

[2026-03-04 23:42:57,282] p37301 {bash_tool.py:20} INFO - Running bash: pwd
[2026-03-04 23:42:57,296] p37301 {4204830944.py:2} INFO - /Users/qing0304/Desktop/GU/DSAN6725/spring-2026-a03-Cynthia2734/mcp-gateway-registry


## Verify Query Classifier

In [5]:
for q in test_questions:
    logger.info(f"[{classify_query(q)}] {q}")

[2026-03-04 23:42:57,301] p37301 {2195511014.py:2} INFO - [dependency] What Python dependencies does this project use?
[2026-03-04 23:42:57,301] p37301 {2195511014.py:2} INFO - [structure] What is the main entry point file for the registry service?
[2026-03-04 23:42:57,301] p37301 {2195511014.py:2} INFO - [structure] What programming languages and file types are used in this repository?
[2026-03-04 23:42:57,302] p37301 {2195511014.py:2} INFO - [auth] How does the authentication flow work, from token validation to user authorization?
[2026-03-04 23:42:57,302] p37301 {2195511014.py:2} INFO - [api] What are all the API endpoints available in the registry service and what scopes do they require?
[2026-03-04 23:42:57,302] p37301 {2195511014.py:2} INFO - [auth] How would you add support for a new OAuth provider (e.g., Okta) to the authentication system?


## Verify Context Retriever

In [6]:
sample_q = test_questions[0]
sample_type = classify_query(sample_q)
sample_ctx = retrieve_context(sample_q, sample_type, CODEBASE_PATH)
print(sample_ctx[:400])

[2026-03-04 23:42:57,305] p37301 {bash_tool.py:20} INFO - Running bash: grep -A 60 '\[project\]' pyproject.toml | head -70
[2026-03-04 23:42:57,316] p37301 {bash_tool.py:20} INFO - Running bash: grep -A 30 'dependencies' auth_server/pyproject.toml | head -35
[2026-03-04 23:42:57,325] p37301 {bash_tool.py:20} INFO - Running bash: python3 -c "import json; d=json.load(open('cli/package.json')); print('name:', d.get('name')); [print(k+': '+v) for k,v in d.get('dependencies',{}).items()]"
[2026-03-04 23:42:57,345] p37301 {retriever.py:134} INFO - retrieve_context [dependency]: 2630 chars


=== pyproject.toml - dependencies ===
[project]
name = "mcp-registry"
version = "0.1.0"
description = "A registry for MCP servers"
readme = "README.md"
requires-python = ">=3.11,<3.14"
dependencies = [
    "aiofiles>=24.1.0",
    "fastapi>=0.115.12",
    "itsdangerous>=2.2.0",
    "jinja2>=3.1.6",
    "mcp>=1.9.3",
    "pydantic>=2.11.3",
    "pydantic-settings>=2.0.0",
    "httpx>=0.27.0",
    "p


## Answer All Test Questions

In [7]:
all_results = []

for i, question in enumerate(test_questions, 1):
    logger.info("=" * 60)
    logger.info(f"Question [{i}/{len(test_questions)}] {question}")
    result = pipeline.answer(question)
    all_results.append(result)
    logger.info(f"Answer [{i}/{len(test_questions)}]:\n{result['answer']}")
    time.sleep(10)

[2026-03-04 23:42:57,351] p37301 {1674143591.py:4} INFO - ============================================================
[2026-03-04 23:42:57,351] p37301 {1674143591.py:5} INFO - Question [1/6] What Python dependencies does this project use?
[2026-03-04 23:42:57,352] p37301 {pipeline.py:66} INFO - Question: What Python dependencies does this project use?
[2026-03-04 23:42:57,352] p37301 {bash_tool.py:20} INFO - Running bash: grep -A 60 '\[project\]' pyproject.toml | head -70
[2026-03-04 23:42:57,362] p37301 {bash_tool.py:20} INFO - Running bash: grep -A 30 'dependencies' auth_server/pyproject.toml | head -35
[2026-03-04 23:42:57,372] p37301 {bash_tool.py:20} INFO - Running bash: python3 -c "import json; d=json.load(open('cli/package.json')); print('name:', d.get('name')); [print(k+': '+v) for k,v in d.get('dependencies',{}).items()]"
[2026-03-04 23:42:57,393] p37301 {retriever.py:134} INFO - retrieve_context [dependency]: 2630 chars
23:42:57 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM compl

## Save Results

In [8]:
output_path = "part1_results.txt"

with open(output_path, "w") as f:
    f.write("Part 1: Code Q&A System with Bash Tools\n")
    f.write("=" * 60 + "\n\n")
    for i, result in enumerate(all_results, 1):
        f.write(f"Question {i}: {result['question']}\n")
        f.write(f"[{result['query_type']}]\n")
        f.write("Answer:\n")
        f.write(f"{result['answer']}\n\n")
        f.write("=" * 60 + "\n\n")

logger.info(f"Saved to {output_path}")

[2026-03-04 23:44:02,536] p37301 {2733539109.py:13} INFO - Saved to part1_results.txt
